# MNIST GAN — RecPulse

A simple Generative Adversarial Network on MNIST. Two MLPs trained adversarially:

- **Generator:** noise (latent=100) → 256 → 512 → 1024 → 784, tanh output (image in [-1, 1]).
- **Discriminator:** 784 → 512 → 256 → 1, LeakyReLU + Dropout, outputs logit per image.

Loss: BCE-with-logits on D's output for both real (target=1) and fake (target=0) batches.
Optimizer: Adam(lr=2e-4, betas=(0.5, 0.999)) — standard DCGAN-style settings.

**Notes:**
- Images normalized to [-1, 1] (matched to G's tanh output).
- D's output is reshaped to 1D `[B]` before BCE — BCE on `[B, 1]` can produce unstable gradients in some chains.
- Intermediate tensors must be retained in named variables (the framework's autograd doesn't refcount); chains like `a.op_x().op_y()` will silently break.

In [ ]:
import sys, time, gc
sys.path.insert(0, '..')

import numpy as np
import recpulse_cuda as rp
from recpulse.module import Module, Linear, Dropout
from recpulse.optim import Adam
from recpulse.data import load_mnist, get_batch

rp.manual_seed(42)
DEVICE = 'cuda'
LATENT = 100

print(f'RecPulse loaded (device={DEVICE}, latent_dim={LATENT})')

## Load MNIST

Re-normalize from [0, 1] to [-1, 1] to match the Generator's tanh output range.

In [ ]:
train_images, _, _, _ = load_mnist('../data/mnist')

arr = train_images.to_numpy() * 2.0 - 1.0
train_images = rp.from_numpy(arr.astype(np.float32))
train_labels = [0] * train_images.shape[0]

print(f'Train: {train_images.shape[0]} images, {train_images.shape[1]} pixels')
print(f'Pixel range: [{arr.min():.2f}, {arr.max():.2f}]')

## Define Generator and Discriminator

In [ ]:
class Generator(Module):
    def __init__(self):
        super().__init__()
        self.fc1 = Linear(LATENT, 256)
        self.fc2 = Linear(256, 512)
        self.fc3 = Linear(512, 1024)
        self.fc4 = Linear(1024, 784)

    def forward(self, z):
        h = self.keep(self.fc1(z))
        h = self.keep(h.op_leaky_relu(0.2))
        h = self.keep(self.fc2(h))
        h = self.keep(h.op_leaky_relu(0.2))
        h = self.keep(self.fc3(h))
        h = self.keep(h.op_leaky_relu(0.2))
        h = self.keep(self.fc4(h))
        return self.keep(h.op_tanh())

class Discriminator(Module):
    def __init__(self):
        super().__init__()
        self.fc1 = Linear(784, 512)
        self.drop1 = Dropout(0.3)
        self.fc2 = Linear(512, 256)
        self.drop2 = Dropout(0.3)
        self.fc3 = Linear(256, 1)

    def forward(self, x):
        h = self.keep(self.fc1(x))
        h = self.keep(h.op_leaky_relu(0.2))
        h = self.keep(self.drop1(h))
        h = self.keep(self.fc2(h))
        h = self.keep(h.op_leaky_relu(0.2))
        h = self.keep(self.drop2(h))
        h = self.keep(self.fc3(h))
        return self.keep(h.reshape([h.shape[0]]))

G = Generator(); G.to(device=DEVICE)
D = Discriminator(); D.to(device=DEVICE)

g_params = sum(t.size for t in G.parameters())
d_params = sum(t.size for t in D.parameters())
print(f'Generator parameters:     {g_params:,}')
print(f'Discriminator parameters: {d_params:,}')

## Setup Training

Adam with lr=2e-4 and betas=(0.5, 0.999) is the standard DCGAN configuration. Lower beta1 (0.5 vs default 0.9) keeps the optimizer responsive to the moving target of GAN training.

In [ ]:
optG = Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH_SIZE = 64
NUM_EPOCHS = 5
num_train = train_images.shape[0]
num_batches = num_train // BATCH_SIZE

ones_dev = rp.from_numpy(np.ones(BATCH_SIZE, dtype=np.float32)).to(device=DEVICE)
zeros_dev = rp.from_numpy(np.zeros(BATCH_SIZE, dtype=np.float32)).to(device=DEVICE)

print(f'Optimizer: Adam (lr=2e-4, betas=(0.5, 0.999))')
print(f'Batch size: {BATCH_SIZE}')
print(f'Batches per epoch: {num_batches}')
print(f'Total epochs: {NUM_EPOCHS}')

## Train

Per batch:
1. **D step** — forward real and fake, compute BCE for each, sum, backward, step D.
2. **G step** — generate fresh fakes, forward through D, compute BCE against `1` (we want D to be fooled), backward, step G.

We zero both networks' gradients before each phase: D's backward computes grads on G's params too (they’re needed for the chain rule), and we don’t want those polluting the G step.

In [ ]:
history = {'d_loss': [], 'g_loss': []}

for epoch in range(NUM_EPOCHS):
    G.train(); D.train()
    d_total = 0.0; g_total = 0.0; n = 0
    start = time.time()

    for b in range(num_batches):
        real_cpu, _ = get_batch(train_images, train_labels, b, BATCH_SIZE)
        if real_cpu is None or real_cpu.shape[0] != BATCH_SIZE:
            break
        real = real_cpu.to(device=DEVICE)

        # ===== D step =====
        D.zero_grad(); G.zero_grad()
        z1 = rp.randn([BATCH_SIZE, LATENT], device=DEVICE)
        fake1 = G(z1)
        d_real = D(real)
        d_fake = D(fake1)
        loss_dr = d_real.op_bce_loss(ones_dev)
        loss_df = d_fake.op_bce_loss(zeros_dev)
        d_loss = loss_dr.op_add(loss_df)
        d_total += float(d_loss.to(device='cpu').sum_all())
        d_loss.backward()
        optD.step()

        # ===== G step =====
        D.zero_grad(); G.zero_grad()
        z2 = rp.randn([BATCH_SIZE, LATENT], device=DEVICE)
        fake2 = G(z2)
        d_fake2 = D(fake2)
        g_loss = d_fake2.op_bce_loss(ones_dev)
        g_total += float(g_loss.to(device='cpu').sum_all())
        g_loss.backward()
        optG.step()

        n += 1
        if b % 50 == 0:
            gc.collect()

    elapsed = time.time() - start
    history['d_loss'].append(d_total / n)
    history['g_loss'].append(g_total / n)
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS}  D={d_total/n:.4f}  G={g_total/n:.4f}  ({elapsed:.0f}s)', flush=True)

## Generate Samples

Generate a 4x4 grid of digits from random latents. Pixel values come back in [-1, 1]; rescale to [0, 1] for display.

In [ ]:
G.eval()
z = rp.randn([16, LATENT], device=DEVICE)
samples = G(z).to(device='cpu').to_numpy()
samples = (samples + 1) / 2
samples = samples.reshape(16, 28, 28).clip(0, 1)

grid = np.zeros((4 * 28, 4 * 28), dtype=np.float32)
for i in range(16):
    r, c = i // 4, i % 4
    grid[r*28:(r+1)*28, c*28:(c+1)*28] = samples[i]

np.save('../data/mnist_gan_samples.npy', grid)
print(f'Saved 4x4 sample grid to data/mnist_gan_samples.npy (shape {grid.shape})')

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(grid, cmap='gray')
    ax.axis('off')
    ax.set_title(f'Generated digits (after {NUM_EPOCHS} epochs)')
    plt.tight_layout()
    plt.savefig('../data/mnist_gan_samples.png', dpi=80, bbox_inches='tight')
    plt.show()
    print('Saved grid image to data/mnist_gan_samples.png')
except ImportError:
    print('matplotlib not available; skipping image render')

## Save Models

In [ ]:
rp.save(G.tracked, '../data/mnist_gan_G.rpt')
rp.save(D.tracked, '../data/mnist_gan_D.rpt')
print('Saved G to data/mnist_gan_G.rpt')
print('Saved D to data/mnist_gan_D.rpt')
print()
print('Loss history:')
print(f"{'Epoch':>5}  {'D loss':>8}  {'G loss':>8}")
for i in range(len(history['d_loss'])):
    print(f"{i+1:5d}  {history['d_loss'][i]:8.4f}  {history['g_loss'][i]:8.4f}")